# Known-stream search: example

How to make matched-filter maps of a known stream in the Euclid DR1 × DES Y6 footprint:

1. the stars and the survey footprint;
2. the galstreams streams crossing the footprint, at 15 < dm < 22;
3. for one stream: its frame, the window of the maps, its isochrones;
4. the map at one distance modulus;
5. the maps over the grid of distance moduli, with the video and the file of maps;
6. maps that follow the distance of the track;
7. the same in one call, for one stream or for all of them.

The maps are star counts in the match filter, in the stream frame (phi1, phi2). No background is subtracted. There is one set of maps per isochrone.

**Phase 2 stub.** Only the import cell runs: the other cells call functions that are not written yet (they raise `NotImplementedError`). Phase 3 makes this notebook run end to end on a validation stream (Section 7 of `docs/dev/known_streams_context.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from streameucliddes.analysis.known_stream_search import (
    known_stream_search_dm_along_track,
    known_stream_search_dm_scan,
    known_stream_search_single_dm,
)
from streameucliddes.catalogs.euclid_des_catalog import get_eucliddes_stars
from streameucliddes.catalogs.galstreams_catalog import (
    galstreams_dm_along_track,
    galstreams_phi12_window,
    load_galstreams,
    select_galstreams_in_footprint,
)
from streameucliddes.catalogs.stream_populations import load_stream_population_table
from streameucliddes.catalogs.survey_footprint import (
    survey_footprint_healpix_mask,
    survey_footprint_phi12_mask,
)
from streameucliddes.isochrone_parameters import known_stream_isochrone_parameters
from streameucliddes.plotting import plot_style
from streameucliddes.plotting.make_videos import make_gif_from_plots, make_mp4
from streameucliddes.plotting.plot_known_stream_search import (
    plot_galstreams_selection,
    plot_known_stream_search_single_dm,
)
from streameucliddes.stream_frames import radec_to_phi12
from streameucliddes.workflows.run_config import RunConfig
from streameucliddes.workflows.run_known_stream_search import (
    run_all_known_stream_searches,
    run_known_stream_search,
    save_known_stream_search_maps,
)

## 1. Stars and footprint

The Euclid × DES stars. Their magnitudes are already dereddened: there is no dust correction below.

The footprint comes from the star counts for now: a HEALPix pixel is in the footprint when it holds a star. A mask is True outside the footprint.

In [ ]:
stars = get_eucliddes_stars()

NSIDE_FOOTPRINT = 512
footprint = survey_footprint_healpix_mask(stars["ra"], stars["dec"], NSIDE_FOOTPRINT, nest=False)

## 2. Known streams in the footprint

The galstreams tracks whose part in the footprint, at 15 < dm < 22, is long enough. Loading galstreams is slow: it is done once.

In [ ]:
mws = load_galstreams()
selection = select_galstreams_in_footprint(mws, footprint, nest=False, dm_range=(15.0, 22.0))

fig, ax = plot_galstreams_selection(selection, mws, footprint, nest=False)
selection

## 3. One stream: frame, window and isochrones

- The frame is the galstreams frame of the track: a gala `GreatCircleICRSFrame`.
- The window covers the track, with margins.
- The isochrones use the age and [Fe/H] of the stream when they are known (`stream_populations.csv`); otherwise the default old, metal-poor grid. Each isochrone gives its own maps: below, the first one.

For the validation, take a DES stream of Shipp et al. 2018 (e.g. ATLAS, Phoenix or Turranburra) with a DES-only footprint.

In [ ]:
row = selection.iloc[0]
track_name, stream_name = row["track_name"], row["stream_name"]

track = mws[track_name]
stream_frame = track.stream_frame
window = galstreams_phi12_window(track)
track_phi12 = radec_to_phi12(track.track.ra.deg, track.track.dec.deg, stream_frame)

population_table = load_stream_population_table()
isochrones = known_stream_isochrone_parameters(stream_name, population_table)
isochrone = isochrones[0]  # one set of maps per isochrone
isochrones

## 4. Map at one distance modulus

The match filter keeps the stars near the isochrone at this distance modulus. They are counted on the phi1/phi2 grid and smoothed. Bins outside the footprint are masked, and the counts are corrected at the footprint edges.

Here, the distance modulus is the median one of the track.

In [ ]:
BIN_SIZE_DEG = 0.1
SMOOTHING_DEG = 0.3
MATCH_FILTER = {
    "name": "des_gr",
    "reference_band": ("des_yr6", "g"),
    "color_bands": [("des_yr6", "r")],
}

footprint_phi12 = survey_footprint_phi12_mask(footprint, stream_frame, window, BIN_SIZE_DEG, nest=False)
dm_track = float(np.nanmedian(galstreams_dm_along_track(track, track_phi12[0])))

result = known_stream_search_single_dm(
    stars, stream_frame, window, dm_track, isochrone, MATCH_FILTER,
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
)
fig, ax = plot_known_stream_search_single_dm(result, track_phi12=track_phi12)

## 5. Scan over distance moduli: figures, video and file of maps

`known_stream_search_dm_scan` gives the maps one distance modulus at a time. Each one is drawn and saved; then the frames make the video, and the maps are saved together in one file. The phi1/phi2 maps are small: keeping them all is fine.

The outputs go to `figures/outputs/` (gitignored).

In [ ]:
isochrone_label = f"age{isochrone['age']:g}_z{isochrone['metallicity']:g}"
output_dir = Path("../figures/outputs/known_stream_search") / track_name / isochrone_label
output_dir.mkdir(parents=True, exist_ok=True)
distance_moduli = np.arange(15.0, 22.0 + 0.05, 0.1)

results = []
for dm, res in known_stream_search_dm_scan(
    stars, stream_frame, window, distance_moduli, isochrone, MATCH_FILTER,
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
):
    fig, ax = plot_known_stream_search_single_dm(res, track_phi12=track_phi12)
    fig.savefig(output_dir / f"{track_name}_dm{dm:.2f}.png", dpi=110)
    plt.close(fig)
    results.append((dm, res))

frames = str(output_dir / f"{track_name}_dm*.png")
make_gif_from_plots(output_dir / f"{track_name}.gif", pattern=frames, duration=300)
make_mp4(output_dir / f"{track_name}.mp4", pattern=frames, duration=300)
save_known_stream_search_maps(
    results, output_dir / f"{track_name}_maps.npz",
    metadata={"track_name": track_name, "isochrone": isochrone},
)

## 6. Following the distance of the track

A stream can have a distance gradient. Here the match filter follows the distance modulus of the track along phi1, plus an offset: one map per offset.

In [ ]:
for dm_offset, res in known_stream_search_dm_along_track(
    stars, track, stream_frame, window, isochrone, MATCH_FILTER,
    dm_offsets=(-0.5, 0.0, 0.5),
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
):
    fig, ax = plot_known_stream_search_single_dm(
        res, track_phi12=track_phi12, title=f"{track_name}, dm offset {dm_offset:+.1f}"
    )

## 7. In one call, for one stream or for all of them

The workflows do steps 3 to 5, for each isochrone of the stream, from a configuration (`configs/known_stream_search.yaml`): for one track, or for every track crossing the footprint.

In [ ]:
config = RunConfig.from_yaml("../configs/known_stream_search.yaml")

summary = run_known_stream_search(
    stars, track_name, config,
    mws=mws, population_table=population_table, footprint_mask=footprint, nest=False,
    output_dir="../figures/outputs/known_stream_search",
)
summary

In [ ]:
table = run_all_known_stream_searches(
    stars, config, nest=False, output_dir="../figures/outputs/known_stream_search",
    mws=mws, population_table=population_table, footprint_mask=footprint,
)
table